<a href="https://colab.research.google.com/github/dionatrafk/model_evaluation/blob/master/Prophet_exec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prophet

In [ ]:
#Importar datasets
!git clone https://github.com/dionatrafk/model_evaluation

## Install Prophet

In [ ]:
!pip install prophet

In [4]:
#define datasets to test
datasets = [
'trace1.csv',
'trace5.csv',
'trace10.csv',
'trace15.csv',
'trace20.csv',
'trace25.csv',
'trace30.csv',
'trace35.csv',
'trace40.csv',
'trace45.csv',
'trace50.csv',
'trace55.csv',
'trace60.csv',
]
results = []

In [5]:
from pandas import read_csv
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from math import sqrt
import numpy as np
import math
import datetime
import warnings
warnings.filterwarnings('ignore')

def prophet_forecast(fileName):
    path = './http_requests_nasa/'
    filename = path + fileName

    print(f"Processing {fileName}...")
    #read the csv file
    dataset = read_csv(filename, header=0, parse_dates=[0], index_col=0)

    # split into train and test sets
    X = dataset.values
    X = X.astype('float32')
    size = int(len(X) * 0.67)
    train, test = X[0:size], X[size:len(X)]
    
    # Get the index for dates
    dataset_dates = dataset.index
    train_dates = dataset_dates[0:size]
    test_dates = dataset_dates[size:len(X)]
    
    predictions = list()

    timer = start = datetime.datetime.now() 
    # walk-forward validation
    for t in range(len(test)):
        # Prepare data for Prophet (requires 'ds' and 'y' columns)
        train_end_idx = size + t
        current_train_data = dataset.iloc[0:train_end_idx]
        
        df_prophet = pd.DataFrame({
            'ds': current_train_data.index,
            'y': current_train_data.values.flatten()
        })
        
        # Create and fit model
        model = Prophet(daily_seasonality=True, weekly_seasonality=False, yearly_seasonality=False)
        model.fit(df_prophet)
        
        # Make forecast for next time step
        future = pd.DataFrame({'ds': [test_dates[t]]})
        forecast = model.predict(future)
        
        yhat = forecast['yhat'].values[0]
        predictions.append(yhat)

    # evaluate forecasts
    yhat = np.asarray(predictions, dtype=np.float32)
    y_test = test.flatten()
    print("Timer: ", datetime.datetime.now() - timer)
    mse = mean_squared_error(y_test, yhat)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_test, yhat)
    r2 = r2_score(y_test, yhat)
    print('R2: %.2f, Testscore: %.2f MSE (%.2f RMSE)' %(r2, mse, rmse))

    # plot some samples
    sample_size = min(100, len(y_test))
    plt.figure(figsize=(10, 4))
    plt.plot(yhat[-sample_size:], label='Predicted')
    plt.plot(y_test[-sample_size:], label='Current')
    plt.legend()
    plt.grid(True)
    plt.ylabel('Requests')
    plt.xlabel('Time')
    plt.title(f'Prophet - {fileName}')
    plt.tight_layout()
    plt.show()
    return {
        "dataset": fileName,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
    }

In [6]:
import pandas as pd

results = []
for fileName in datasets:
    result = prophet_forecast(fileName)
    results.append(result)

if results:
    summary_df = pd.DataFrame(results)
    summary_df = summary_df[["dataset", "mae", "mse", "rmse", "r2"]]
    display(summary_df.round({"mae": 3, "mse": 3, "rmse": 3, "r2": 4}))